In [1]:
import os
import time
import sys
import io

# sys.path.append(os.path.abspath(os.path.join(os.path.abspath(os.getcwd()), '..')))
from jawm import Process

In [2]:
# Clear any global state
Process.reset_stop()

passed = 0
failed = 0

In [3]:
print(">>> Test 1: Basic Inline Script Execution")
time.sleep(0.5)
try:
    proc1 = Process(
        name="basic_hello",
        script="""#!/bin/bash
    echo 'Hello JAWM!'
    """,
        logs_directory="logs_test"
    )
    proc1.execute()
    Process.wait(proc1.hash)
    assert proc1.get_exitcode() == "0", "❌ Basic execution failed"
    print("✅ Passed: Basic Inline Script")
    passed += 1
except Exception as e:
    print(f"❌ Failed: — {e}")
    failed += 1

>>> Test 1: Basic Inline Script Execution


[2025-07-24 10:06:29] - INFO - basic_hello|yadeema :: Executing process basic_hello locally.
[2025-07-24 10:06:29] - INFO - basic_hello|yadeema :: Log folder for process basic_hello: /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/basic_hello_20250724_100629_yadeema
[2025-07-24 10:06:29] - INFO - basic_hello|yadeema :: Preparing base script for process basic_hello
[2025-07-24 10:06:29] - INFO - basic_hello|yadeema :: Waiting for process basic_hello (yadeema) to complete...(triggered by Process.wait)
[2025-07-24 10:06:29] - INFO - basic_hello|yadeema :: Process basic_hello started with PID: 4136511
[2025-07-24 10:06:29] - INFO - basic_hello|yadeema :: Process basic_hello (PID: 4136511) is still running...
[2025-07-24 10:06:34] - INFO - basic_hello|yadeema :: Process basic_hello completed with exit code: 0
[2025-07-24 10:06:34] - INFO - basic_hello|yadeema :: Waiting for process basic_hello (yadeema) has completed (triggered by Process.wait)


Process.wait | INFO :: Wait completed for 1 process(es).
✅ Passed: Basic Inline Script


In [4]:
print("\n>>> Test 2: Dependency Handling")
time.sleep(0.5)
try:
    proc2a = Process(
        name="step_a",
        script="""#!/bin/bash
    echo 'Step A done'
    """,
        logs_directory="logs_test"
    )
    proc2b = Process(
        name="step_b",
        script="""#!/bin/bash
    echo 'Step B done'
    """,
        depends_on=["step_a"],
        logs_directory="logs_test"
    )
    proc2a.execute()
    proc2b.execute()
    Process.wait(["step_a", "step_b"])
    assert proc2a.get_exitcode() == "0", "❌ step_a failed"
    assert proc2b.get_exitcode() == "0", "❌ step_b failed"
    print("✅ Passed: Dependency Handling")
    passed += 1
except Exception as e:
    print(f"❌ Failed: — {e}")
    failed += 1


>>> Test 2: Dependency Handling


[2025-07-24 10:06:35] - INFO - step_a|ng93t66 :: Executing process step_a locally.
[2025-07-24 10:06:35] - INFO - step_a|ng93t66 :: Log folder for process step_a: /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/step_a_20250724_100635_ng93t66
[2025-07-24 10:06:35] - INFO - step_a|ng93t66 :: Preparing base script for process step_a
[2025-07-24 10:06:35] - INFO - step_b|sign9ot :: Waiting for dependency process step_a (ng93t66) to finish before executing step_b (sign9ot)
[2025-07-24 10:06:35] - INFO - step_a|ng93t66 :: Process step_a started with PID: 4136520
[2025-07-24 10:06:35] - INFO - step_a|ng93t66 :: Process step_a completed with exit code: 0
[2025-07-24 10:06:35] - INFO - step_b|sign9ot :: Executing process step_b locally.
[2025-07-24 10:06:35] - INFO - step_b|sign9ot :: Log folder for process step_b: /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/step_b_20250724_100635_sign9ot
[2025-07-24 10:06:35] - INFO - step_b|sign9ot :: Prepar

Process.wait | INFO :: Wait completed for 2 process(es).
✅ Passed: Dependency Handling


In [5]:
print("\n>>> Test 3: Retry Mechanism")
time.sleep(0.5)
try:
    proc3 = Process(
        name="retry_test",
        script="""#!/bin/bash
    fakecommandddddd
    """,
        retries=1,
        logs_directory="logs_test"
    )
    try:
        proc3.execute()
        Process.wait(proc3.hash)
    except RuntimeError:
        pass
    assert proc3.get_exitcode() != "0", "❌ Retry test unexpectedly succeeded"
    time.sleep(0.5)
    print("✅ Passed: Retry Mechanism")
    passed += 1
except Exception as e:
    print(f"❌ Failed: — {e}")
    failed += 1

# Clear any global state
Process.reset_stop()


>>> Test 3: Retry Mechanism


[2025-07-24 10:06:35] - INFO - retry_test|b21tdkw :: Executing process retry_test locally.
[2025-07-24 10:06:35] - INFO - retry_test|b21tdkw :: Log folder for process retry_test: /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/retry_test_20250724_100635_b21tdkw
[2025-07-24 10:06:35] - INFO - retry_test|b21tdkw :: Preparing base script for process retry_test
[2025-07-24 10:06:35] - INFO - retry_test|b21tdkw :: Waiting for process retry_test (b21tdkw) to complete...(triggered by Process.wait)
[2025-07-24 10:06:35] - INFO - retry_test|b21tdkw :: Process retry_test started with PID: 4136527
[2025-07-24 10:06:35] - INFO - retry_test|b21tdkw :: Process retry_test (PID: 4136527) is still running...
[2025-07-24 10:06:40] - INFO - retry_test|b21tdkw :: Process retry_test completed with exit code: 127
[2025-07-24 10:06:40] - ERROR - retry_test|b21tdkw :: Attempt 1 for process retry_test failed with exit code 127
[2025-07-24 10:06:40] - INFO - retry_test|b21tdkw :: Retryin

Process.wait | INFO :: Wait completed for 1 process(es).
✅ Passed: Retry Mechanism


In [6]:
print("\n>>> Test 4: Output, Error, and Command Log Check")
time.sleep(0.5)
try:
    output = proc2b.get_output()
    error = proc2b.get_error()
    command = proc2b.get_command()
    assert output is not None, "❌ Output log missing"
    assert error is not None, "❌ Error log missing"
    assert command is not None, "❌ Command log missing"
    print("✅ Passed: Log Retrieval Checks")
    passed += 1
except Exception as e:
    print(f"❌ Failed: — {e}")
    failed += 1


>>> Test 4: Output, Error, and Command Log Check
✅ Passed: Log Retrieval Checks


In [7]:
print("\n>>> Test 5: Process Registry Summary")
time.sleep(0.5)
try:
    all_procs = Process.list_all()
    assert all(p["finished"] for p in all_procs), "❌ Some processes not marked finished"
    print(f"✅ Passed: {len(all_procs)} Process(es) tracked and marked finished")
    passed += 1
except Exception as e:
    print(f"❌ Failed: — {e}")
    failed += 1


>>> Test 5: Process Registry Summary
✅ Passed: 4 Process(es) tracked and marked finished


In [8]:
print("\n>>> Test 6: Script Variable Substitution")
time.sleep(0.5)
try:
    proc6 = Process(
        name="var_subst",
        script="""#!/bin/bash
    echo "Job name is {{APPNAME}}"
    """,
        var={"APPNAME": "JAWM-Test"},
        logs_directory="logs_test"
    )
    proc6.execute()
    Process.wait(proc6.hash)
    out6 = proc6.get_output()
    assert "JAWM-Test" in out6, "❌ var not substituted correctly"
    print("✅ Passed: Script Variable Substitution")
    passed += 1
except Exception as e:
    print(f"❌ Failed: — {e}")
    failed += 1


>>> Test 6: Script Variable Substitution


[2025-07-24 10:06:48] - INFO - var_subst|pvm4yaa :: Executing process var_subst locally.
[2025-07-24 10:06:48] - INFO - var_subst|pvm4yaa :: Log folder for process var_subst: /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/var_subst_20250724_100648_pvm4yaa
[2025-07-24 10:06:48] - INFO - var_subst|pvm4yaa :: Preparing base script for process var_subst
[2025-07-24 10:06:48] - INFO - var_subst|pvm4yaa :: Waiting for process var_subst (pvm4yaa) to complete...(triggered by Process.wait)
[2025-07-24 10:06:48] - INFO - var_subst|pvm4yaa :: Process var_subst started with PID: 4136548
[2025-07-24 10:06:48] - INFO - var_subst|pvm4yaa :: Process var_subst completed with exit code: 0
[2025-07-24 10:06:48] - INFO - var_subst|pvm4yaa :: Waiting for process var_subst (pvm4yaa) has completed (triggered by Process.wait)


Process.wait | INFO :: Wait completed for 1 process(es).
✅ Passed: Script Variable Substitution


In [9]:
print("\n>>> Test 7: Script Variable File Substitution")
time.sleep(0.5)

try:
    os.makedirs("data_test", exist_ok=True)
    
    # Create a simple .rc file
    with open("data_test/vars.rc", "w") as f:
        f.write("GREETING=Hello from file\n")
    
    proc7 = Process(
        name="file_vars",
        script="""#!/bin/bash
    echo "{{GREETING}}"
    """,
        var_file="data_test/vars.rc",
        logs_directory="logs_test"
    )
    proc7.execute()
    Process.wait(proc7.hash)
    out7 = proc7.get_output()
    assert "Hello from file" in out7, "❌ var_file substitution failed"
    print("✅ Passed: Script Variable File Substitution")
    passed += 1
except Exception as e:
    print(f"❌ Failed: — {e}")
    failed += 1



>>> Test 7: Script Variable File Substitution


[2025-07-24 10:06:48] - INFO - file_vars|nu8u6oa :: Executing process file_vars locally.
[2025-07-24 10:06:48] - INFO - file_vars|nu8u6oa :: Log folder for process file_vars: /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/file_vars_20250724_100648_nu8u6oa
[2025-07-24 10:06:48] - INFO - file_vars|nu8u6oa :: Preparing base script for process file_vars
[2025-07-24 10:06:48] - INFO - file_vars|nu8u6oa :: Waiting for process file_vars (nu8u6oa) to complete...(triggered by Process.wait)
[2025-07-24 10:06:48] - INFO - file_vars|nu8u6oa :: Process file_vars started with PID: 4136550
[2025-07-24 10:06:48] - INFO - file_vars|nu8u6oa :: Process file_vars completed with exit code: 0
[2025-07-24 10:06:48] - INFO - file_vars|nu8u6oa :: Waiting for process file_vars (nu8u6oa) has completed (triggered by Process.wait)


Process.wait | INFO :: Wait completed for 1 process(es).
✅ Passed: Script Variable File Substitution


In [10]:
print("\n>>> Test 8: Skipped Process using `when=False`")
time.sleep(0.5)
try:
    proc8 = Process(
        name="skip_this",
        script="""#!/bin/bash
    echo 'Should not run' > skip.txt
    """,
        when=False,
        logs_directory="logs_test"
    )
    proc8.execute()
    assert proc8.finished_event.is_set(), "❌ Skipped process did not mark finished"
    assert not os.path.exists(os.path.join(proc8.log_path, "skip.txt")), "❌ Script ran despite when=False"
    print("✅ Passed: Conditional Skip with `when=False`")
    passed += 1
except Exception as e:
    print(f"❌ {e}")
    failed += 1

Process.reset_stop()


>>> Test 8: Skipped Process using `when=False`


[2025-07-24 10:06:49] - INFO - skip_this|bk47bpv :: Process skip_this skipped because 'when' condition was not fulfilled!


✅ Passed: Conditional Skip with `when=False`


In [11]:
print("\n>>> Test 9: Process Cloning with `copy()`")
time.sleep(0.5)
try:
    original = Process(
        name="original_proc",
        script="""#!/bin/bash
    echo 'Original'
    """,
        logs_directory="logs_test"
    )
    clone = original.copy(name="cloned_proc", script="""#!/bin/bash
    echo 'Cloned'
    """)
    original.execute()
    clone.execute()
    Process.wait(["original_proc", "cloned_proc"])
    assert "original_proc" in Process.registry
    assert "cloned_proc" in Process.registry
    print("✅ Passed: Process Copy and Execution")
    passed += 1
except Exception as e:
    print(f"❌ Failed: {e}")
    failed += 1


>>> Test 9: Process Cloning with `copy()`


[2025-07-24 10:06:49] - INFO - original_proc|bwithi6 :: Executing process original_proc locally.
[2025-07-24 10:06:49] - INFO - original_proc|bwithi6 :: Log folder for process original_proc: /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/original_proc_20250724_100649_bwithi6
[2025-07-24 10:06:49] - INFO - original_proc|bwithi6 :: Preparing base script for process original_proc
[2025-07-24 10:06:49] - INFO - cloned_proc|iflkoai :: Executing process cloned_proc locally.
[2025-07-24 10:06:49] - INFO - cloned_proc|iflkoai :: Log folder for process cloned_proc: /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/cloned_proc_20250724_100649_iflkoai
[2025-07-24 10:06:49] - INFO - cloned_proc|iflkoai :: Preparing base script for process cloned_proc
[2025-07-24 10:06:49] - INFO - original_proc|bwithi6 :: Process original_proc started with PID: 4136552
[2025-07-24 10:06:49] - INFO - original_proc|bwithi6 :: Waiting for process original_proc (bwithi6) 

Process.wait | INFO :: Wait completed for 2 process(es).
✅ Passed: Process Copy and Execution


In [12]:
print("\n>>> Test 10: Retry Parameter Overrides")
time.sleep(0.5)

try:
    proc10 = Process(
        name="retry_override_test",
        script="""#!/bin/bash
    fakecomandddddd
    """,
        retries=1,
        manager="slurm",
        manager_slurm={"--mem": "1G", "--time": "00:01:00"},
        retry_overrides={
            1: {"manager_slurm": {"--mem": "+100%", "--time": "+60"}},
            2: {"manager_slurm": {"--mem": "3G", "--time": "00:05:00"}},
        },
        logs_directory="logs_test"
    )
    
    try:
        proc10.execute()
        Process.wait(proc10.hash)
    except RuntimeError:
        pass
    
    # Check the .slurm file to see if override applied
    slurm_log = proc10.get_slurm()
    assert "--mem=2G" in slurm_log or "--mem 2G" in slurm_log, "❌ Retry override did not apply"
    print("✅ Passed: Retry Parameter Overrides")
    passed += 1
except Exception as e:
    print(f"❌ {e}")
    failed += 1
    
Process.reset_stop()



>>> Test 10: Retry Parameter Overrides


[2025-07-24 10:06:50] - INFO - retry_override_test|6zlbg38 :: Executing process retry_override_test in Slurm
[2025-07-24 10:06:50] - INFO - retry_override_test|6zlbg38 :: Log folder for process retry_override_test: /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/retry_override_test_20250724_100650_6zlbg38
[2025-07-24 10:06:50] - INFO - retry_override_test|6zlbg38 :: Generating Slurm job script for process: retry_override_test
[2025-07-24 10:06:50] - INFO - retry_override_test|6zlbg38 :: Waiting for process retry_override_test (6zlbg38) to complete...(triggered by Process.wait)
[2025-07-24 10:06:50] - INFO - retry_override_test|6zlbg38 :: Preparing base script for process retry_override_test
[2025-07-24 10:06:50] - INFO - retry_override_test|6zlbg38 :: Submitting process retry_override_test with slurm command: sbatch --mem 1G --time 00:01:00 --output /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/retry_override_test_20250724_100650_6zlbg3

Process.wait | INFO :: Wait completed for 1 process(es).
✅ Passed: Retry Parameter Overrides


Exception in thread Thread-13 (monitor_process):
Traceback (most recent call last):
  File "/opt/python/3.10.8/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/opt/python/3.10.8/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/opt/python/3.10.8/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/jawm/_process_slurm.py", line 224, in monitor_process
    raise RuntimeError(f"Process {self.name} in Slurm failed after {total_attempts} attempts.")
RuntimeError: Process retry_override_test in Slurm failed after 2 attempts.


In [13]:
print("\n>>> Test 14: Basic Slurm Job Execution")
time.sleep(0.5)

try:
    proc14 = Process(
        name="slurm_hello",
        script="""#!/bin/bash
    echo 'Hello from Slurm!'
    """,
        manager="slurm",
        manager_slurm={
            "--partition": "cluster",     # <- Change if needed
            "--time": "00:01:00"
        },
        logs_directory="logs_test"
    )
    
    proc14.execute()
    Process.wait(proc14.hash)
    
    exitcode = proc14.get_exitcode()
    output = proc14.get_output()
    
    assert exitcode == "0:0", f"❌ Slurm job failed with exit code {exitcode}"
    assert "Hello from Slurm!" in output, "❌ Slurm script did not produce expected output"
    print("✅ Passed: Basic Slurm Job Execution")
    passed += 1
except Exception as e:
    print(f"❌ {e}")
    failed += 1


>>> Test 14: Basic Slurm Job Execution


[2025-07-24 10:07:31] - INFO - slurm_hello|2d9o7ni :: Executing process slurm_hello in Slurm
[2025-07-24 10:07:31] - INFO - slurm_hello|2d9o7ni :: Log folder for process slurm_hello: /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/slurm_hello_20250724_100731_2d9o7ni
[2025-07-24 10:07:31] - INFO - slurm_hello|2d9o7ni :: Generating Slurm job script for process: slurm_hello
[2025-07-24 10:07:31] - INFO - slurm_hello|2d9o7ni :: Preparing base script for process slurm_hello
[2025-07-24 10:07:31] - INFO - slurm_hello|2d9o7ni :: Waiting for process slurm_hello (2d9o7ni) to complete...(triggered by Process.wait)
[2025-07-24 10:07:31] - INFO - slurm_hello|2d9o7ni :: Submitting process slurm_hello with slurm command: sbatch --partition cluster --time 00:01:00 --output /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/slurm_hello_20250724_100731_2d9o7ni/slurm_hello.output --error /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/

Process.wait | INFO :: Wait completed for 1 process(es).
✅ Passed: Basic Slurm Job Execution


In [14]:
print("\n>>> Test 15: Slurm + Apptainer Container Execution")
time.sleep(0.5)

try:
    # Set the path to your Apptainer image
    apptainer_image = "/nexus/posix0/MAGE-flaski/service/images/python.sif"  # <-- UPDATE this to a valid path!
    
    proc15 = Process(
        name="slurm_apptainer_test",
        script="""#!/bin/bash
    echo "Apptainer+Slurm works!"
    """,
        manager="slurm",
        environment="apptainer",
        container=apptainer_image,
        manager_slurm={
            "--partition": "cluster",      # <-- Change if needed
            "--time": "00:01:00"
        },
        logs_directory="logs_test"
    )
    
    proc15.execute()
    Process.wait(proc15.hash)
    
    exitcode = proc15.get_exitcode()
    output = proc15.get_output()
    
    assert exitcode == "0:0", f"❌ Slurm+Apptainer job failed with exit code {exitcode}"
    assert "Apptainer+Slurm works!" in output, "❌ Output missing or incorrect"
    print("✅ Passed: Slurm Execution with Apptainer Container")
    passed += 1
except Exception as e:
    print(f"❌ {e}")
    failed += 1


>>> Test 15: Slurm + Apptainer Container Execution


[2025-07-24 10:07:52] - INFO - slurm_apptainer_test|jnfregs :: Executing process slurm_apptainer_test in Slurm
[2025-07-24 10:07:52] - INFO - slurm_apptainer_test|jnfregs :: Log folder for process slurm_apptainer_test: /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/slurm_apptainer_test_20250724_100752_jnfregs
[2025-07-24 10:07:52] - INFO - slurm_apptainer_test|jnfregs :: Generating Slurm job script for process: slurm_apptainer_test
[2025-07-24 10:07:52] - INFO - slurm_apptainer_test|jnfregs :: Waiting for process slurm_apptainer_test (jnfregs) to complete...(triggered by Process.wait)
[2025-07-24 10:07:52] - INFO - slurm_apptainer_test|jnfregs :: Preparing base script for process slurm_apptainer_test
[2025-07-24 10:07:52] - INFO - slurm_apptainer_test|jnfregs :: Submitting process slurm_apptainer_test with slurm command: sbatch --partition cluster --time 00:01:00 --output /nexus/posix0/MAGE-flaski/service/posit/home/hamin/JAWM/draft/logs_test/slurm_apptainer_te

Process.wait | INFO :: Wait completed for 1 process(es).
✅ Passed: Slurm Execution with Apptainer Container


In [15]:
print("\n===== TEST SUMMARY =====")
print(f"✅ Passed: {passed}")
print(f"❌ Failed: {failed}")

if failed == 0:
    print("🎉 All tests passed!")
else:
    print("⚠️ Some tests failed — review the output.")


===== TEST SUMMARY =====
✅ Passed: 12
❌ Failed: 0
🎉 All tests passed!
